**Setup of Patients List DB**

1- **Setup on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Patients_List"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [

    # Primary Key
    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING",
        mode="REQUIRED"
    ),

    # Basic Information
    bigquery.SchemaField("Patient_Name", "STRING"),
    bigquery.SchemaField("Phone_Number", "STRING"),
    bigquery.SchemaField("National_ID", "STRING"),
    bigquery.SchemaField("Address_Area", "STRING"),
    bigquery.SchemaField("Gender", "STRING"),

    # Measurements
    bigquery.SchemaField("Ht", "FLOAT"),
    bigquery.SchemaField("BMI", "FLOAT"),
    bigquery.SchemaField("BMI_Category", "STRING"),

    # Occupation
    bigquery.SchemaField("Occupation_Type", "STRING"),
    bigquery.SchemaField("Occupation_Details", "STRING"),

    # Lifestyle
    bigquery.SchemaField("Smoker", "BOOLEAN"),

    # Medical Information
    bigquery.SchemaField("Allergies", "STRING"),
    bigquery.SchemaField(
        "Chronic_Conditions",
        "STRING",
        mode="REPEATED"
    ),
    bigquery.SchemaField("Blood_Group", "STRING"),

    # Emergency Contact
    bigquery.SchemaField("Emergency_Contact", "STRING"),

    # Dates
    bigquery.SchemaField("Profile_Creation_Date", "DATE"),
    bigquery.SchemaField("First_Visit_Date", "DATE"),
    bigquery.SchemaField("Last_Visit_Date", "DATE")
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

table.clustering_fields = [
    "National_ID",
    "Patient_U_ID"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Patients_List


In [2]:
import sys
import psycopg2
sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS patient_no_seq;

""")
# =========================================
# CREATE TABLE QUERY
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Patients_List (

    -- Auto Generated Primary Key
    Patient_U_ID TEXT PRIMARY KEY
    DEFAULT (
    'PAT_' ||
    LPAD(nextval('patient_no_seq')::TEXT, 6, '0')),


    -- Basic Information
    Patient_Name TEXT,
    Phone_Number TEXT,

    -- UNIQUE National ID
    National_ID TEXT UNIQUE,
    Address_Area TEXT,
    Gender TEXT,

    -- Measurements
    Ht FLOAT,
    BMI FLOAT,
    BMI_Category TEXT,

    -- Occupation
    Occupation_Type TEXT,
    Occupation_Details TEXT,

    -- Lifestyle
    Smoker BOOLEAN,

    -- Medical Information
    Allergies TEXT,
    Chronic_Conditions TEXT[],
    Blood_Group TEXT,

    -- Emergency Contact
    Emergency_Contact TEXT,

    -- Dates
    Profile_Creation_Date DATE,
    First_Visit_Date DATE,
    Last_Visit_Date DATE
);
"""

# =========================================
# EXECUTE QUERY
# =========================================

cursor.execute(create_table_query)

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Patients_List")
print("=================================")

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Patients_List


**Sync BigQuery to Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2
import numpy as np

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# IMPORT CONNECTIONS
# =========================================

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ FROM BIGQUERY
# =========================================

query = """

SELECT *

FROM `depihealthnux.Depihealthnux_Main.Patients_List`

ORDER BY Patient_U_ID

"""

df = client.query(query).to_dataframe()

print("=================================")
print("FIRST 3 ROWS FROM BIGQUERY")
print("=================================")
print(df.head(3))

print(f"\nRows Retrieved: {len(df)}")

# =========================================
# FIX ARRAY COLUMNS
# =========================================

if "Chronic_Conditions" in df.columns:

    df["Chronic_Conditions"] = df["Chronic_Conditions"].apply(

        lambda x:
        x.tolist()

        if isinstance(x, np.ndarray)

        else x

    )

# =========================================
# FIX NaN / NaT
# =========================================

df = df.astype(object).where(
    pd.notnull(df),
    None
)

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CLEAR TABLE
# =========================================

cursor.execute("""

DELETE FROM Patients_List;

""")

# =========================================
# BUILD RECORDS
# =========================================

records = []

for _, row in df.iterrows():

    records.append(

        (

            row["Patient_U_ID"],
            row["Patient_Name"],
            row["Phone_Number"],
            row["National_ID"],
            row["Address_Area"],
            row["Gender"],

            row["Ht"],
            row["BMI"],
            row["BMI_Category"],

            row["Occupation_Type"],
            row["Occupation_Details"],

            row["Smoker"],

            row["Allergies"],
            row["Chronic_Conditions"],
            row["Blood_Group"],

            row["Emergency_Contact"],

            row["Profile_Creation_Date"],
            row["First_Visit_Date"],
            row["Last_Visit_Date"]

        )

    )

# =========================================
# DEBUG
# =========================================

if records:

    print("\n=================================")
    print("FIRST RECORD")
    print("=================================")
    print(records[0])

# =========================================
# INSERT DATA
# =========================================

if records:

    execute_values(

        cursor,

        """

        INSERT INTO Patients_List (

            Patient_U_ID,
            Patient_Name,
            Phone_Number,
            National_ID,
            Address_Area,
            Gender,

            Ht,
            BMI,
            BMI_Category,

            Occupation_Type,
            Occupation_Details,

            Smoker,

            Allergies,
            Chronic_Conditions,
            Blood_Group,

            Emergency_Contact,

            Profile_Creation_Date,
            First_Visit_Date,
            Last_Visit_Date

        )

        VALUES %s

        """,

        records,

        page_size=1000

    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'patient_no_seq',
    GREATEST(
        COALESCE(
            (
                SELECT MAX(
                    CAST(
                        REPLACE(
                            Patient_U_ID,
                            'PAT_',
                            ''
                        ) AS INTEGER
                    )
                )
                FROM Patients_List
            ),
            0
        ),
        1
    ),
    true
);

""")

# =========================================
# COMMIT
# =========================================

conn.commit()

print("\n=================================")
print(f"Inserted {len(records)} Rows")
print("=================================")

# =========================================
# VERIFY
# =========================================

cursor.execute("""

SELECT *

FROM Patients_List

ORDER BY Patient_U_ID

LIMIT 3

""")

print("\n=================================")
print("FIRST 3 ROWS FROM POSTGRES")
print("=================================")

for row in cursor.fetchall():
    print(row)

# =========================================
# CLOSE
# =========================================

cursor.close()
conn.close()

FIRST 3 ROWS FROM BIGQUERY
  Patient_U_ID   Patient_Name Phone_Number     National_ID Address_Area  \
0   PAT_000001  Sedeek Ashraf  01061650353  29408042101778         Giza   
1   PAT_000005    Tamer Ahmed    010161251        23123112        Cairo   
2   PAT_000006   Hamada Selim    012021122                        Cairo   

  Gender     Ht   BMI BMI_Category    Occupation_Type Occupation_Details  \
0   Male    NaN  None         None               None               None   
1   Male  170.2  None         None  Office Based Work                      
2   Male  180.0  None         None  Office Based Work                      

   Smoker Allergies    Chronic_Conditions Blood_Group   Emergency_Contact  \
0   False         -  [J45.90, E10, E78.4]        None                None   
1    True                           [E10]          A+  Ahmed - 1021021021   
2    True            [E10, E55, D50, I25]          A-                       

  Profile_Creation_Date First_Visit_Date Last_Visit_Date  

**Sync Postgres to BigQuery**

In [1]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ FROM POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT *

FROM Patients_List

ORDER BY Patient_U_ID

"""

df = pd.read_sql(query, conn)

conn.close()

print("=================================")
print("FIRST 3 ROWS FROM POSTGRES")
print("=================================")
print(df.head(3))

print(f"\nRows Retrieved: {len(df)}")

# =========================================
# SKIP IF EMPTY
# =========================================

if df.empty:

    print("=================================")
    print("No Patients Found")
    print("BigQuery Sync Skipped")
    print("=================================")

else:

    # =========================================
    # RESTORE BIGQUERY COLUMN NAMES
    # =========================================

    df.columns = [

        "Patient_U_ID",
        "Patient_Name",
        "Phone_Number",
        "National_ID",
        "Address_Area",
        "Gender",
        "Ht",
        "BMI",
        "BMI_Category",
        "Occupation_Type",
        "Occupation_Details",
        "Smoker",
        "Allergies",
        "Chronic_Conditions",
        "Blood_Group",
        "Emergency_Contact",
        "Profile_Creation_Date",
        "First_Visit_Date",
        "Last_Visit_Date"

    ]

    # =========================================
    # DATE COLUMNS
    # =========================================

    date_cols = [

        "Profile_Creation_Date",
        "First_Visit_Date",
        "Last_Visit_Date"

    ]

    for col in date_cols:

        df[col] = pd.to_datetime(
            df[col]
        ).dt.date

    # =========================================
    # UPLOAD
    # =========================================

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )

    job = client.load_table_from_dataframe(
        df,
        "depihealthnux.Depihealthnux_Main.Patients_List",
        job_config=job_config
    )

    job.result()

    print("\n=================================")
    print(f"Uploaded {len(df)} Rows")
    print("=================================")

    verify_df = client.query("""

    SELECT *

    FROM `depihealthnux.Depihealthnux_Main.Patients_List`

    LIMIT 3

    """).to_dataframe()

    print("\nFIRST 3 ROWS FROM BIGQUERY")
    print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_14292\2962923921.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


FIRST 3 ROWS FROM POSTGRES
  patient_u_id   patient_name phone_number     national_id address_area  \
0   PAT_000001  Sedeek Ashraf  01061650353  29408042101778         Giza   
1   PAT_000005    Tamer Ahmed    010161251        23123112        Cairo   
2   PAT_000006   Hamada Selim    012021122                        Cairo   

  gender     ht   bmi bmi_category    occupation_type occupation_details  \
0   Male    NaN  None         None               None               None   
1   Male  170.2  None         None  Office Based Work                      
2   Male  180.0  None         None  Office Based Work                      

   smoker allergies    chronic_conditions blood_group   emergency_contact  \
0   False         -  [J45.90, E10, E78.4]        None                None   
1    True                           [E10]          A+  Ahmed - 1021021021   
2    True            [E10, E55, D50, I25]          A-                       

  profile_creation_date first_visit_date last_visit_date  